In [ ]:
import os
from dotenv import load_dotenv
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.vector_stores.pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from llama_index.core import StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline


In [ ]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")


In [ ]:
gemini_embedding = GoogleGenAIEmbedding(
    model_name="models/text-embedding-004",
    api_key=api_key,
)
pc = Pinecone(api_key=pinecone_api_key)


In [19]:
reader = SimpleDirectoryReader(input_dir="data", required_exts=[".txt"])
documents = reader.load_data()
print(f"Documents: {documents}")

Documents: [Document(id_='64966a81-4d17-4c1b-8e48-10712a8b9e2e', embedding=None, metadata={'file_path': 'c:\\TrainingDeliverables\\Python\\LatestDemos\\llamaindexdemos\\data\\notes.txt', 'file_name': 'notes.txt', 'file_type': 'text/plain', 'file_size': 17, 'creation_date': '2025-08-21', 'last_modified_date': '2025-08-21'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='HI\r\nHello there!!', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')]


In [ ]:
pc.create_index(
    name="pindex",
    dimension=768,  # Gemini text-embedding-004 outputs 768 dims
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)


In [21]:
pinecone_index = pc.Index("pindex")

In [22]:
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

In [27]:
# rebuild storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)
# load index
index = VectorStoreIndex.from_documents(documents, storage_context)

2025-08-22 08:57:56,899 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


In [ ]:
# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=25, chunk_overlap=0),
        TitleExtractor(),
        GoogleGenAIEmbedding(
            model_name="models/text-embedding-004",
            api_key=api_key,
        ),
    ],
    vector_store=vector_store,
)


In [25]:
pipeline.run(documents)
pinecone_index.describe_index_stats()

Parsing nodes: 0it [00:00, ?it/s]
0it [00:00, ?it/s]
Generating embeddings: 0it [00:00, ?it/s]


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 2}},
 'total_vector_count': 2,
 'vector_type': 'dense'}

In [28]:
all_ids = list(pinecone_index.list())
print("All vector IDs:", all_ids)

All vector IDs: [['64966a81-4d17-4c1b-8e48-10712a8b9e2e#80fb9056-afa6-4404-9427-9efadcad6685', '64966a81-4d17-4c1b-8e48-10712a8b9e2e#b2da87f6-f22b-4709-92b8-9becb0e654b6', 'aae5c5a3-3f11-4e01-950c-008c3dd4a180#25e9c510-e0f1-43b1-af29-f7762b069fd9']]


In [26]:
query_engine = index.as_query_engine()
response = query_engine.query("do I have Ravi in the text file?") 
print(f"Response: {response}")

2025-08-22 08:57:46,948 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-22 08:57:48,798 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Ravi is not present in the text file.
